In [ ]:
# Skriptas 'flags' reikšmių skaičiavimui įrašų json failuose.
# Skaičiuojama visiems įrašams nurodytiems įrašų saraše tipo EXCEL_NAME = "visi_zive_irasai.xlsx",
# išskyrus įrašus, kuriems  df[df["tag"] != "9999"]
# https://chatgpt.com/c/697347e3-01ac-8327-af6a-c28ba234de2e

# Ieškoma 'flags' reikšmių:
#   "PSEUDO_ANNOTATED",
#   "FOR_EXTERNAL_ANNOTATING",
#   "EXTERNALLY_ANNOTATED",
#   "FULLY_ANNOTATED_PROFESSIONALLY"

# loc_fname - įrašo failo vardas (be plėtinio) 

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, TypedDict, Any, Set
from collections import Counter
import pandas as pd
import sys

# funkcija skaito Excel su įrašų sąrašu ir meta duomenimis 
def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    print("Skaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    print("Atrinkta įrašų:", len(names))
    return names, df

class EcgJsonSummary(TypedDict):
    flags_01: Dict[str, int]
    noises_annotated_count: int
    noises_annotated_raw_count: int
    intervals: List[Tuple[int, int]]


def _read_json_data(json_path: Path) -> Optional[Dict[str, Any]]:
    """
    Reads JSON file and returns parsed dict.
    Returns None if file does not exist, JSON is invalid, or top-level is not a dict.
    """
    if not json_path.exists():
        return None

    try:
        with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError):
        return None

    if not isinstance(data, dict):
        return None

    return data


def _extract_raw_flags_set(data: Dict[str, Any]) -> Optional[Set[str]]:
    """
    Extracts flags from JSON data and returns them as a set of strings.
    Returns None if 'flags' exists but is not a list (invalid structure).
    """
    raw_flags = data.get("flags")

    if raw_flags is None:
        raw_flags_list: List[str] = []
    elif isinstance(raw_flags, list):
        raw_flags_list = [str(x) for x in raw_flags]
    else:
        return None  # invalid structure

    return set(raw_flags_list)


def _extract_valid_intervals(data: Dict[str, Any]) -> Optional[Tuple[int, List[Tuple[int, int]]]]:
    """
    Extracts noises_annotated from JSON data.
    Returns (raw_count, valid_intervals).
    Returns None if 'noises_annotated' exists but is not a list (invalid structure).
    """
    items = data.get("noises_annotated")

    if items is None:
        return 0, []

    if not isinstance(items, list):
        return None  # invalid structure

    raw_count = len(items)
    valid_intervals: List[Tuple[int, int]] = []

    for it in items:
        try:
            valid_intervals.append((int(it["startIndex"]), int(it["endIndex"])))
        except (KeyError, TypeError, ValueError):
            continue

    return raw_count, valid_intervals


def count_flags_from_ecg_json(
    json_path: Path,
    known_flags: Optional[List[str]] = None,
    return_intervals: bool = False,
) -> Optional[EcgJsonSummary]:
    """
    Returns a summary of JSON:
      - flags presence as 0/1 per flag type (based on known_flags)
      - count of noises_annotated intervals (valid + raw)
      - optionally parsed intervals

    If file does not exist or JSON structure is invalid -> returns None.
    """
    data = _read_json_data(json_path)
    if data is None:
        return None

    raw_flags_set = _extract_raw_flags_set(data)
    if raw_flags_set is None:
        return None

    # Keep your debug print if you want it
    print(f"{json_path.stem}, flags: {raw_flags_set}")

    known_flags = known_flags or [
        "PSEUDO_ANNOTATED",
        "FOR_EXTERNAL_ANNOTATING",
        "EXTERNALLY_ANNOTATED",
        "FULLY_ANNOTATED_PROFESSIONALLY",
    ]

    flags_01 = {flag: (1 if flag in raw_flags_set else 0) for flag in known_flags}

    extracted = _extract_valid_intervals(data)
    if extracted is None:
        return None
    raw_count, valid_intervals = extracted

    result: EcgJsonSummary = {
        "flags_01": flags_01,
        "noises_annotated_count": len(valid_intervals),
        "noises_annotated_raw_count": raw_count,
        "intervals": valid_intervals if return_intervals else [],
    }
    return result

# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from project_util import find_project_root_by_name


# === Konfigūracija ==============================================================

PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

LIST_DIR = PROJECT_ROOT / "DATA_ORIG"/ "ecg_zive_npy"
REC_DIR = PROJECT_ROOT / "DATA_ORIG"/ "ecg_zive_npy"

# EXCEL_NAME = "visi_zive_irasai_test.xlsx"
# EXCEL_NAME = "visi_zive_irasai_testui.xlsx"
EXCEL_NAME = "visi_zive_irasai.xlsx"

loc_loc_fnames, df_meta = read_filenames_from_excel(LIST_DIR / EXCEL_NAME)
print(loc_loc_fnames)

flags_total = Counter()   # sums 0/1 across all records => "how many records have this flag"
records_with_noises = 0
records_without_noises = 0
total_records = 0

# optional diagnostics
missing_or_invalid = 0  # if your loader returns None for missing/invalid JSON

for i, loc_fname in enumerate(loc_loc_fnames, start=1):
    fpath = REC_DIR / loc_fname

    try:
        result = count_flags_from_ecg_json(fpath.with_suffix(".json"))
    except Exception as exc:
        raise ValueError(f"Failed to load {loc_fname}: {exc}") from exc

    # If you kept "None means missing/invalid" semantics:
    if result is None:
        missing_or_invalid += 1
        continue

    total_records += 1

    # 1) aggregate flags
    flags_01: Dict[str, int] = result.get("flags_01", {})
    flags_total.update(flags_01)  # adds 0/1 per flag into totals

    # 2) aggregate noises_annotated presence
    n_noises = int(result.get("noises_annotated_count", 0))
    if n_noises > 0:
        records_with_noises += 1
    else:
        records_without_noises += 1

# ---- summary printout ----
print("\n=== SUMMARY ===")
print(f"Processed records (valid JSON): {total_records}")
if missing_or_invalid:
    print(f"Skipped (missing/invalid JSON): {missing_or_invalid}")

print("\nFlags (how many records have each flag):")
for flag, cnt in sorted(flags_total.items()):
    print(f"  {flag}: {cnt}")

print("\nNoises annotated:")
print(f"  Records WITH noises_annotated intervals: {records_with_noises}")
print(f"  Records WITHOUT noises_annotated intervals: {records_without_noises}")




PROJECT ROOT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA
Skaitomas Excel: %s /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_ORIG/ecg_zive_npy/visi_zive_irasai.xlsx
Atrinkta įrašų: 1085
['1000_1.npy', '1001_1.npy', '1001_2.npy', '1001_3.npy', '1001_4.npy', '1001_5.npy', '1001_6.npy', '1001_7.npy', '1001_8.npy', '1002_0.npy', '1002_1.npy', '1002_2.npy', '1002_3.npy', '1002_4.npy', '1002_5.npy', '1002_6.npy', '1003_0.npy', '1003_1.npy', '1004_0.npy', '1004_1.npy', '1004_2.npy', '1005_0.npy', '1005_1.npy', '1005_10.npy', '1005_11.npy', '1005_12.npy', '1005_13.npy', '1005_14.npy', '1005_15.npy', '1005_16.npy', '1005_17.npy', '1005_2.npy', '1005_3.npy', '1005_4.npy', '1005_5.npy', '1005_6.npy', '1005_7.npy', '1005_8.npy', '1005_9.npy', '1006_0.npy', '1006_1.npy', '1006_2.npy', '1006_3.npy', '1007_0.npy', '1007_1.npy', '1008_0.npy', '1008_1.npy', '1008_10.npy', '1008_11.npy', '1008_12.npy', '1008_13.